<a href="https://colab.research.google.com/github/Nana0191/spark_assignment/blob/main/Spark_handson.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PySprak ETL Task: Sales Data Processing

### Description
Utilize PySpark skills to clean and transform the sales dataset from CSV file `sales.csv`, enrich it with product category info from the second file `products.csv`, and store the result as a Parquet file for future analytics.


In [13]:
!rm -f sales.csv products.csv
!wget -q -O sales.csv https://raw.githubusercontent.com/Nana0191/spark_assignment/main/sales.csv
!wget -q -O products.csv https://raw.githubusercontent.com/Nana0191/spark_assignment/main/products.csv
!head -3 sales.csv products.csv

==> sales.csv <==
transaction_id,product_id,date,quantity,unit_price
1,1028,2023-05-31,3,30.36
2,1007,2023-12-11,2,86.91

==> products.csv <==
product_id,product_name,category
1001,Product_0,Adapters
1002,Product_1,Peripherals



### Requirements
**1. Extract and Load data files into PySprak Session**

In [14]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("SalesETL").getOrCreate()

In [15]:
sales_df = spark.read.csv("sales.csv", header=True, inferSchema=True)
products_df = spark.read.csv("products.csv", header=True, inferSchema=True)

sales_df.printSchema()
sales_df.show(5)
products_df.show(5)
print("sales rows:", sales_df.count(), "| products rows:", products_df.count())

root
 |-- transaction_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- date: date (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)

+--------------+----------+----------+--------+----------+
|transaction_id|product_id|      date|quantity|unit_price|
+--------------+----------+----------+--------+----------+
|             1|      1028|2023-05-31|       3|     30.36|
|             2|      1007|2023-12-11|       2|     86.91|
|             3|      1073|2023-10-19|       2|     59.57|
|             4|      1072|2023-03-31|       5|     68.52|
|             5|      1012|2023-10-07|       3|     44.71|
+--------------+----------+----------+--------+----------+
only showing top 5 rows
+----------+------------+-----------+
|product_id|product_name|   category|
+----------+------------+-----------+
|      1001|   Product_0|   Adapters|
|      1002|   Product_1|Peripherals|
|      1003|   Product_2|     Cables|
|   

**2. Transform, Clean & Enrich the data by applying column transformations and joining the tables**

In [16]:
sales_clean = (
    sales_df
    .dropDuplicates(["transaction_id"])
    .dropna()
    .filter((F.col("quantity") > 0) & (F.col("unit_price") > 0))
)
print("rows after cleaning:", sales_clean.count())

rows after cleaning: 100000


In [17]:
sales_clean = (
    sales_clean
    .withColumn("date", F.to_date("date", "yyyy-MM-dd"))
    .withColumn("quantity", F.col("quantity").cast("int"))
    .withColumn("unit_price", F.col("unit_price").cast("double"))
    .withColumn("total_amount", F.round(F.col("quantity") * F.col("unit_price"), 2))
    .withColumn("year", F.year("date"))
    .withColumn("month", F.month("date"))
)
sales_clean.show(5)

+--------------+----------+----------+--------+----------+------------+----+-----+
|transaction_id|product_id|      date|quantity|unit_price|total_amount|year|month|
+--------------+----------+----------+--------+----------+------------+----+-----+
|           148|      1040|2023-02-13|       5|     99.13|      495.65|2023|    2|
|           463|      1021|2023-11-28|       4|     89.74|      358.96|2023|   11|
|           471|      1099|2023-11-07|       5|     37.05|      185.25|2023|   11|
|           496|      1100|2023-06-02|       3|     55.96|      167.88|2023|    6|
|           833|      1052|2023-07-23|       3|     16.01|       48.03|2023|    7|
+--------------+----------+----------+--------+----------+------------+----+-----+
only showing top 5 rows


In [18]:
products_clean = (
    products_df
    .dropDuplicates(["product_id"])
    .withColumn("category", F.trim(F.col("category")))
)

In [19]:
final_df = (
    sales_clean
    .join(products_clean, on="product_id", how="left")
    .select("transaction_id", "product_id", "product_name", "category",
            "date", "year", "month", "quantity", "unit_price", "total_amount")
)
final_df.show(5)
print("final rows:", final_df.count())
print("rows without category:", final_df.filter(F.col("category").isNull()).count())

+--------------+----------+------------+-----------+----------+----+-----+--------+----------+------------+
|transaction_id|product_id|product_name|   category|      date|year|month|quantity|unit_price|total_amount|
+--------------+----------+------------+-----------+----------+----+-----+--------+----------+------------+
|           148|      1040|  Product_39|Accessories|2023-02-13|2023|    2|       5|     99.13|      495.65|
|           463|      1021|  Product_20|Accessories|2023-11-28|2023|   11|       4|     89.74|      358.96|
|           471|      1099|  Product_98|Accessories|2023-11-07|2023|   11|       5|     37.05|      185.25|
|           496|      1100|  Product_99|Electronics|2023-06-02|2023|    6|       3|     55.96|      167.88|
|           833|      1052|  Product_51|   Adapters|2023-07-23|2023|    7|       3|     16.01|       48.03|
+--------------+----------+------------+-----------+----------+----+-----+--------+----------+------------+
only showing top 5 rows
fina

**3. Load – Save the clean data as a Parquet file**

In [20]:
final_df.write.mode("overwrite").parquet("output/sales_enriched.parquet")

In [21]:
check_df = spark.read.parquet("output/sales_enriched.parquet")
check_df.printSchema()
check_df.show(5)
print("rows saved:", check_df.count())

root
 |-- transaction_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- date: date (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- total_amount: double (nullable = true)

+--------------+----------+------------+-----------+----------+----+-----+--------+----------+------------+
|transaction_id|product_id|product_name|   category|      date|year|month|quantity|unit_price|total_amount|
+--------------+----------+------------+-----------+----------+----+-----+--------+----------+------------+
|           148|      1040|  Product_39|Accessories|2023-02-13|2023|    2|       5|     99.13|      495.65|
|           463|      1021|  Product_20|Accessories|2023-11-28|2023|   11|       4|     89.74|      358.96|
|           471|      1099|  Product_98|Acce